<a href="https://colab.research.google.com/github/chamarairesh1982/LearnPython/blob/main/chamara_Week_2_Data_Preprocessing_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
import pandas as pd  # a library for data analysis, provides a convenient way to load, clean, and analyze data.This is a useful tool to work with tabular data. Most of the EDA tasks can be accomplished using Pandas.
import numpy as np   # a library for numerical computing

import matplotlib.pyplot as plt  #a library for creating visualizations, provides a wide range of plotting functions, including line plots, bar charts, and pie charts.
import seaborn as sns  # a library for creating statistical visualizations, builds on Matplotlib and provides a number of high-level functions for creating attractive and informative visualizations.
# Scikit-learn provides a wide range of ML algorithms, including supervised, unsupervised learning, and NLP.
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from imblearn.over_sampling import SMOTE

from google.colab import files

# Load the dataset
*  Pandas provides you with a number of methods in order to read data from a given dataset. For example, you can mount Google drive or upload the dataset to a public GitHub repository,and call it.
*  It does not matter what the format of your dataset is (e.g. csv, xlsx). By locating the address of your file (either on your hard drive or somewhere on the web) you can easily read from your dataset and build your Data Frame out of it.
*   To proceed with the below example, follow these steps.
Step 1: Download the dataset from Kaggle. Use this link: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset?resource=download

* Run the below code to Upload dataset to Colab. A file selection window will appear. Choose: WA_Fn-UseC_-HR-Employee-Attrition.csv (the downloaded file) from your local machine.

In [ ]:
uploaded = files.upload()

In [2]:
# Load the Dataset
df = pd.read_csv("https://drive.google.com/file/d/1gpDdgcgsojN2I3Pb6YZG2Gga1W3cXr8I/view?usp=drive_link/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df.head()

HTTPError: HTTP Error 401: Unauthorized

# Data Profiling

In [ ]:
# 1
df.info()  #read dataset information
df.shape  #Shape
df.dtypes #Data Types
df.describe() # Summary Statistics
df.isnull().sum()  #Missing Values
df.duplicated().sum()  #Duplicate Records
df.nunique()  #Unique Values

# Exploratory Data Analysis

In [ ]:
sns.countplot(x='Attrition', data=df)   #Target Distribution
plt.show()

In [ ]:
df.hist(figsize=(15,10))  #Numerical Distribution
plt.show()

In [ ]:
sns.boxplot(x=df["MonthlyIncome"])   #Boxplot
plt.show()

In [ ]:
plt.figure(figsize=(15,10))   #Correlation Heatmap
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.show()

In [ ]:
# 3. Data Cleaning
df.loc[10:20,"MonthlyIncome"]=np.nan  #Introduce Missing Values (For teaching purpose only)
df.loc[40:45,"Age"]=np.nan

In [ ]:
# Check Missing Values
df.isnull().sum()

In [ ]:
imputer = SimpleImputer(strategy="median") #Imputation
df[["Age","MonthlyIncome"]] = imputer.fit_transform(df[["Age","MonthlyIncome"]])

In [ ]:
df.drop_duplicates(inplace=True)  # Remove Duplicates

In [ ]:
sns.boxplot(df["MonthlyIncome"])  #Check Outliers
plt.show()

In [ ]:
# 4. Data Integration and Create another dataframe
department = pd.DataFrame({
"Department":["Sales","Research & Development","Human Resources"],
"Location":["Building A","Building B","Building C"]
})

In [ ]:
merged = pd.merge(df,department,on="Department")  #Merge
merged.head()

In [ ]:
# 5. Feature Engineering
merged["IncomePerYear"]=merged["MonthlyIncome"]*12   #Income per Year
merged["ExperienceRatio"]=merged["YearsAtCompany"]/merged["Age"]       #Experience Ratio
merged["AgeGroup"]=pd.cut(     #Age Groups
merged["Age"],
bins=[18,30,40,50,60],
labels=["Young","Adult","Middle","Senior"]
)

In [ ]:
# 6. Data Transformation
encoder=LabelEncoder()    #Label Encoding
merged["Attrition"]=encoder.fit_transform(merged["Attrition"])

In [ ]:
merged=pd.get_dummies(  #One-Hot Encoding
merged,
columns=["BusinessTravel"],
drop_first=True
)

In [ ]:
scaler=StandardScaler()    #Standardization
merged[["Age","MonthlyIncome"]]=scaler.fit_transform(
merged[["Age","MonthlyIncome"]]
)

In [ ]:
# Min-Max Scaling
minmax=MinMaxScaler()
merged[["DistanceFromHome"]]=minmax.fit_transform(
merged[["DistanceFromHome"]]
)

In [ ]:
# 7. Data Reduction
X=merged.drop("Attrition",axis=1)  #Feature Selection
X=pd.get_dummies(X)
y=merged["Attrition"]

selector=SelectKBest(score_func=chi2,k=10)
X_new=selector.fit_transform(abs(X),y)
print(X_new.shape)

In [ ]:
pca=PCA(n_components=5)   # PCA
X_pca=pca.fit_transform(X_new)
print(X_pca.shape)

In [ ]:
# 8. Handling Class Imbalance
pd.Series(y).value_counts()   # Check Distribution

In [ ]:
smote=SMOTE(random_state=42)  # Apply SMOTE
X_balanced,y_balanced=smote.fit_resample(X_pca,y)

In [ ]:
pd.Series(y_balanced).value_counts().plot(kind="bar") #New Distribution

In [ ]:
# 9. Data Splitting
X_train,X_test,y_train,y_test=train_test_split(
X_balanced,
y_balanced,
test_size=0.2,
random_state=42,
stratify=y_balanced
)

In [ ]:
#Check Shapes
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)
print("Preprocessing Completed Successfully!")